In [ ]:
!pip install ultralytics


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os

zip_path = "/content/drive/MyDrive/ToText/synthetic_emnist_yolo.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/synthetic_emnist_yolo")

os.listdir("/content/synthetic_emnist_yolo")


Mounted at /content/drive


['synthetic_emnist_yolo']

In [ ]:
import yaml

CHAR_MAP = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz!?.,:;'\"()[]{}<>@#$%^&*+-=/\\|_"
class_names = list(CHAR_MAP)
nc=len(class_names)

data_yaml = {
    'path': 'synthetic_emnist_yolo/synthetic_emnist_yolo',
    'train': 'images/train',
    'val': 'images/val',
    'nc': nc,
    'names': class_names
}

with open("emnist.yaml", "w") as f:
    yaml.dump(data_yaml, f)


In [ ]:
from ultralytics import YOLO

model_name = 'yolo11n.pt'

# Main idea is to have a hard save every 25 epochs, better if the training crashes but makes it a bit slower
iterations = 6
for i in range(len(iterations)):
    if i == 0:
        model = YOLO(model_name=model_name)
    else:
        model = YOLO('runs/train_emnist_yolo/yolov10s_emnist/weights/last.pt')
    model.train(
        data='emnist.yaml',
        epochs=25,
        imgsz=640,
        batch=8, # Can rep;ace here for -1 for RAM usage
        workers=1,
        project="runs/train_emnist_yolo",
        name="yolov10s_emnist",
        device=0 ,
        flipud=0.2,        
        fliplr=0.5,        
        hsv_h=0.02,        
        hsv_s=0.7,         
        hsv_v=0.4,         
        degrees=10.0,      
        translate=0.1,     
        scale=0.5,         
        shear=2.0,         
        perspective=0.001, 
        mosaic=1.0,     # can be deleted for better RAM usage   
        mixup=0.1,       # can be deleted for better RAM usage    
        cache=False  # longer training time but better ram usage

)


In [ ]:
import shutil
from google.colab import files

output_dir = '/content/runs/train_emnist_yolo/yolov10s_emnist'
zip_path = '/content/yolov10s_emnist_results.zip'

shutil.make_archive(base_name=zip_path.replace('.zip', ''), format='zip', root_dir=output_dir)

files.download(zip_path)
